# Mini projet 2 - PokéAPI

In [1]:
import requests

In [2]:
url = "https://pokeapi.co/api/v2/pokemon/pikachu"
data = requests.get(url).json()
data

{'id': 25,
 'name': 'pikachu',
 'base_experience': 112,
 'height': 4,
 'is_default': True,
 'order': 35,
 'weight': 60,
 'abilities': [{'is_hidden': False,
   'slot': 1,
   'ability': {'name': 'static',
    'url': 'https://pokeapi.co/api/v2/ability/9/'}},
  {'is_hidden': True,
   'slot': 3,
   'ability': {'name': 'lightning-rod',
    'url': 'https://pokeapi.co/api/v2/ability/31/'}}],
 'past_abilities': [{'generation': {'name': 'generation-iv',
    'url': 'https://pokeapi.co/api/v2/generation/4/'},
   'abilities': [{'is_hidden': True, 'slot': 3, 'ability': None}]}],
 'forms': [{'name': 'pikachu',
   'url': 'https://pokeapi.co/api/v2/pokemon-form/25/'}],
 'game_indices': [{'game_index': 84,
   'version': {'name': 'red', 'url': 'https://pokeapi.co/api/v2/version/1/'}},
  {'game_index': 84,
   'version': {'name': 'blue', 'url': 'https://pokeapi.co/api/v2/version/2/'}},
  {'game_index': 84,
   'version': {'name': 'yellow',
    'url': 'https://pokeapi.co/api/v2/version/3/'}},
  {'game_index'

In [3]:
import time
import random
import asyncio
import aiohttp

URL = "https://pokeapi.co/api/v2"

## Q1 - Gérer les erreurs

- 429 (trop de requêtes) et 500/502/503/504 (erreur serveur) : c'est temporaire, on attend et on réessaie. J'attends de plus en plus longtemps à chaque essai (1s, 2s, 4s...). Pour le 429 je regarde aussi le header `Retry-After` qui dit combien de temps attendre.
- 400, 401, 403, 404 : ça sert à rien de réessayer, la requête est fausse (mauvais id, pas de token...). On affiche l'erreur et on s'arrête.
- Timeout ou problème réseau : on réessaie aussi.

In [4]:
def safe_get(url, retries=5):
    for i in range(retries):
        try:
            r = requests.get(url, timeout=10)
        except requests.RequestException as e:
            print("Problème réseau :", e)
            time.sleep(2 ** i)
            continue

        if r.status_code == 200:
            return r.json()

        if r.status_code in (429, 500, 502, 503, 504):
            attente = int(r.headers.get("Retry-After", 2 ** i))
            print(f"Erreur {r.status_code}, nouvel essai dans {attente}s")
            time.sleep(attente)
            continue

        erreurs = {400: "Requête invalide", 401: "Token absent ou invalide", 403: "Accès interdit", 404: "Introuvable (id non valide)"}
        print(f"Erreur {r.status_code} : {erreurs.get(r.status_code, 'erreur inconnue')}")
        return None

    print("Abandon après", retries, "essais")
    return None

In [5]:
print(safe_get(f"{URL}/pokemon/25")["name"])
print(safe_get(f"{URL}/pokemon/pokemonquiexistepas"))

pikachu
Erreur 404 : Introuvable (id non valide)
None


## Stats d'un Pokémon

In [6]:
STATS = ["hp", "attack", "defense", "special-attack", "special-defense", "speed"]


def get_pokemon(nom):
    nom = str(nom).lower().strip()
    if nom == "":
        print("Il faut entrer un nom ou un id")
        return None
    data = safe_get(f"{URL}/pokemon/{nom}")
    if data is None:
        return None
    p = {s["stat"]["name"]: s["base_stat"] for s in data["stats"]}
    p["name"] = data["name"]
    p["id"] = data["id"]
    p["types"] = [t["type"]["name"] for t in data["types"]]
    p["taille"] = data["height"] / 10   # en m
    p["poids"] = data["weight"] / 10    # en kg
    p["total"] = sum(p[s] for s in STATS)
    return p


def afficher_stats():
    nom = input("Nom ou id du pokémon : ")
    p = get_pokemon(nom)
    if p is None:
        return
    print(f"{p['name']} (n°{p['id']}) - types : {', '.join(p['types'])} - {p['taille']} m, {p['poids']} kg")
    for s in STATS:
        print(f"  {s:<16} {p[s]}")
    print(f"  {'total':<16} {p['total']}")


afficher_stats()

pikachu (n°25) - types : electric - 0.4 m, 6.0 kg
  hp               35
  attack           55
  defense          40
  special-attack   50
  special-defense  50
  speed            90
  total            320


## Comparer 2 Pokémon

In [7]:
def comparer():
    p1 = get_pokemon(input("Pokémon 1 : "))
    p2 = get_pokemon(input("Pokémon 2 : "))
    if p1 is None or p2 is None:
        return

    print(f"{'':<16}{p1['name']:>12}{p2['name']:>12}")
    for s in STATS + ["total"]:
        print(f"{s:<16}{p1[s]:>12}{p2[s]:>12}")

    for s in ["hp", "attack"]:
        if p1[s] == p2[s]:
            print(f"{s} : égalité")
        else:
            print(f"{s} : {p1['name'] if p1[s] > p2[s] else p2['name']} est plus fort")


comparer()

                  charmander   charizard
hp                        39          78
attack                    52          84
defense                   43          78
special-attack            60         109
special-defense           50          85
speed                     65         100
total                    309         534
hp : charizard est plus fort
attack : charizard est plus fort


## Stats par type

In [8]:
def stats_type():
    t = input("Type (fire, water, dragon...) : ")
    data = safe_get(f"{URL}/type/{t.lower().strip()}")
    if data is None:
        return
    noms = [x["pokemon"]["name"] for x in data["pokemon"]]
    pv = [get_pokemon(n)["hp"] for n in noms]   # 1 requête par pokémon -> c'est lent (voir Q2)
    print(f"{len(noms)} pokémons de type {t}, PV moyen : {sum(pv) / len(pv):.1f}")


debut = time.time()
stats_type()
print(f"{time.time() - debut:.1f}s")

109 pokémons de type fire, PV moyen : 72.5
42.6s


## Combat

À chaque tour les 2 pokémons attaquent une fois. Dégâts = attaque / défense de l'adversaire × 20, avec un peu d'aléatoire. Au bout de 5 tours celui qui a fait le plus de dégâts gagne.

In [9]:
def combat():
    p1 = get_pokemon(input("Pokémon 1 : "))
    p2 = get_pokemon(input("Pokémon 2 : "))
    if p1 is None or p2 is None:
        return

    degats = {p1["name"]: 0, p2["name"]: 0}
    for tour in range(1, 6):
        for att, dfs in [(p1, p2), (p2, p1)]:
            d = round(att["attack"] / dfs["defense"] * 20 * random.uniform(0.85, 1))
            degats[att["name"]] += d
            print(f"Tour {tour} : {att['name']} fait {d} dégâts à {dfs['name']}")

    print("Total :", degats)
    if degats[p1["name"]] == degats[p2["name"]]:
        print("Égalité")
    else:
        print("Gagnant :", max(degats, key=degats.get))


combat()

Tour 1 : eevee fait 17 dégâts à squirtle
Tour 1 : squirtle fait 17 dégâts à eevee
Tour 2 : eevee fait 16 dégâts à squirtle
Tour 2 : squirtle fait 18 dégâts à eevee
Tour 3 : eevee fait 15 dégâts à squirtle
Tour 3 : squirtle fait 19 dégâts à eevee
Tour 4 : eevee fait 16 dégâts à squirtle
Tour 4 : squirtle fait 19 dégâts à eevee
Tour 5 : eevee fait 16 dégâts à squirtle
Tour 5 : squirtle fait 17 dégâts à eevee
Total : {'eevee': 80, 'squirtle': 90}
Gagnant : squirtle


## Q2 - Beaucoup de requêtes avec asyncio

En séquentiel on attend chaque réponse avant d'envoyer la suivante. Avec asyncio + aiohttp on envoie plein de requêtes en même temps et on attend les réponses en parallèle.

Pour que ça tienne avec des millions de requêtes :
- un `Semaphore` limite le nombre de requêtes en même temps (sinon on sature la machine et on se prend des 429)
- une seule `ClientSession` pour réutiliser les connexions
- même logique de retry que Q1 mais avec `await asyncio.sleep()` pour ne pas bloquer les autres requêtes
- on compte les résultats au lieu de tout garder en mémoire

Dans Jupyter on peut faire `await` directement dans une cellule.

In [10]:
async def fetch(session, sem, url, retries=5):
    async with sem:
        for i in range(retries):
            try:
                async with session.get(url) as r:
                    if r.status == 200:
                        return await r.json()
                    if r.status not in (429, 500, 502, 503, 504):
                        return None   # 400, 401, 403, 404 -> on abandonne
            except aiohttp.ClientError:
                pass
            await asyncio.sleep(2 ** i)
        return None


async def fetch_all(urls, max_simultane=100):
    sem = asyncio.Semaphore(max_simultane)
    async with aiohttp.ClientSession() as session:
        return await asyncio.gather(*[fetch(session, sem, u) for u in urls])

Même calcul que pour les types mais en parallèle :

In [11]:
debut = time.time()
data = safe_get(f"{URL}/type/fire")
res = await fetch_all([x["pokemon"]["url"] for x in data["pokemon"]])
pv = [p["stats"][0]["base_stat"] for p in res if p]
print(f"{len(pv)} pokémons fire, PV moyen : {sum(pv) / len(pv):.1f}")
print(f"{time.time() - debut:.1f}s")

109 pokémons fire, PV moyen : 72.5
0.8s


### Simulation de charge

api.example.com n'existe pas vraiment donc je lance un petit serveur local avec les routes /data1, /data2, /data3. Il met entre 20 et 80 ms à répondre et renvoie parfois des 429 ou des 503 pour tester les retry.

In [12]:
from aiohttp import web

nb_appels = 0

async def route(request):
    global nb_appels
    nb_appels += 1
    await asyncio.sleep(random.uniform(0.02, 0.08))
    x = random.random()
    if x < 0.05:
        return web.Response(status=429)
    if x < 0.08:
        return web.Response(status=503)
    return web.json_response({"route": request.path, "valeur": random.randint(0, 100)})

app = web.Application()
app.router.add_get("/{nom}", route)
runner = web.AppRunner(app)
await runner.setup()
await web.TCPSite(runner, "localhost", 8765).start()

ROUTES = ["http://localhost:8765/data1", "http://localhost:8765/data2", "http://localhost:8765/data3"]

In [13]:
N = 10_000
urls = [ROUTES[i % 3] for i in range(N)]

nb_appels = 0
debut = time.time()
res = await fetch_all(urls, max_simultane=100)
duree = time.time() - debut

print(f"{N} requêtes en {duree:.1f}s ({N / duree:.0f} req/s)")
print("réussies :", sum(r is not None for r in res))
print("appels reçus par le serveur (avec les retry) :", nb_appels)

10000 requêtes en 18.8s (532 req/s)
réussies : 10000
appels reçus par le serveur (avec les retry) : 10811


Toutes les requêtes passent malgré les erreurs grâce aux retry. En séquentiel 10 000 × 50 ms ça ferait plus de 8 minutes.

## Lister et trier les Pokémon

In [14]:
res = await fetch_all([f"{URL}/pokemon/{i}" for i in range(1, 152)])
pokedex = []
for data in res:
    if data:
        p = {s["stat"]["name"]: s["base_stat"] for s in data["stats"]}
        p["name"] = data["name"]
        p["id"] = data["id"]
        p["types"] = [t["type"]["name"] for t in data["types"]]
        p["taille"] = data["height"] / 10
        p["poids"] = data["weight"] / 10
        p["total"] = sum(p[s] for s in STATS)
        pokedex.append(p)
print(len(pokedex), "pokémons chargés")


def lister(type_=None, tri="total", croissant=False, n=10):
    liste = [p for p in pokedex if type_ is None or type_ in p["types"]]
    liste = sorted(liste, key=lambda p: p[tri], reverse=not croissant)
    for p in liste[:n]:
        print(f"{p['id']:>4} {p['name']:<12} {', '.join(p['types']):<18} {p['taille']:>5} m {p['poids']:>6} kg  pv={p['hp']} att={p['attack']} def={p['defense']} vit={p['speed']} total={p['total']}")


print("\nLes plus lourds :")
lister(tri="poids")
print("\nLes pokémons eau les plus rapides :")
lister(type_="water", tri="speed")

151 pokémons chargés

Les plus lourds :
 143 snorlax      normal               2.1 m  460.0 kg  pv=160 att=110 def=65 vit=30 total=540
  76 golem        rock, ground         1.4 m  300.0 kg  pv=80 att=120 def=130 vit=45 total=495
 130 gyarados     water, flying        6.5 m  235.0 kg  pv=95 att=125 def=79 vit=81 total=540
 131 lapras       water, ice           2.5 m  220.0 kg  pv=130 att=85 def=80 vit=60 total=535
  95 onix         rock, ground         8.8 m  210.0 kg  pv=35 att=45 def=160 vit=70 total=385
 149 dragonite    dragon, flying       2.2 m  210.0 kg  pv=91 att=134 def=95 vit=80 total=600
  59 arcanine     fire                 1.9 m  155.0 kg  pv=90 att=110 def=80 vit=95 total=555
  91 cloyster     water, ice           1.5 m  132.5 kg  pv=50 att=95 def=180 vit=70 total=525
  68 machamp      fighting             1.6 m  130.0 kg  pv=90 att=130 def=80 vit=55 total=505
 150 mewtwo       psychic              2.0 m  122.0 kg  pv=106 att=110 def=90 vit=130 total=680

Les pokémons ea

## Q3 - Cache d'1h

Je garde les réponses dans un dictionnaire `url -> (heure d'enregistrement, données)`. Si l'url est déjà dedans et a moins d'1h, je renvoie la donnée sans appeler l'API. Sinon j'appelle l'API et je mets à jour le cache.

Ça évite de rappeler l'API pour la même chose (moins de charge sur le serveur) et c'est beaucoup plus rapide.

In [15]:
cache = {}

async def cached_fetch(session, sem, url, duree=3600):
    if url in cache:
        heure, data = cache[url]
        if time.time() - heure < duree:
            return data
    data = await fetch(session, sem, url)
    cache[url] = (time.time(), data)
    return data


session = aiohttp.ClientSession()
sem = asyncio.Semaphore(100)

In [16]:
# 1er appel : API / 2e et 3e : cache
for i in range(3):
    debut = time.time()
    await cached_fetch(session, sem, f"{URL}/pokemon/charizard")
    print(f"appel {i + 1} : {(time.time() - debut) * 1000:.2f} ms")

appel 1 : 54.80 ms
appel 2 : 0.00 ms
appel 3 : 0.00 ms


Pour montrer l'expiration sans attendre 1h je mets une durée de 2 secondes. Le serveur local renvoie une valeur aléatoire à chaque vrai appel, donc on voit si ça vient du cache ou pas :

In [17]:
cache.clear()
print(await cached_fetch(session, sem, ROUTES[0], duree=2))
print(await cached_fetch(session, sem, ROUTES[0], duree=2), "<- même valeur, ça vient du cache")
await asyncio.sleep(2.5)
print(await cached_fetch(session, sem, ROUTES[0], duree=2), "<- nouvelle valeur, le cache a expiré")

{'route': '/data1', 'valeur': 27}
{'route': '/data1', 'valeur': 27} <- même valeur, ça vient du cache
{'route': '/data1', 'valeur': 95} <- nouvelle valeur, le cache a expiré


Avec 10 000 requêtes vers les 3 mêmes routes, le serveur n'est presque plus appelé :

In [18]:
cache.clear()
nb_appels = 0
debut = time.time()
for i in range(10_000):
    await cached_fetch(session, sem, ROUTES[i % 3])
print(f"10000 requêtes en {time.time() - debut:.2f}s, appels reçus par le serveur : {nb_appels}")

await session.close()

10000 requêtes en 0.17s, appels reçus par le serveur : 3


## Front Streamlit

La cellule écrit le fichier `app_pokemon.py`. Pour le lancer : `streamlit run app_pokemon.py`

Le cache d'1h côté Streamlit est fait avec `@st.cache_data(ttl=3600)`.

In [19]:
%%writefile app_pokemon.py
import asyncio
import random
import time

import aiohttp
import requests
import streamlit as st

URL = "https://pokeapi.co/api/v2"
STATS = ["hp", "attack", "defense", "special-attack", "special-defense", "speed"]


def safe_get(url, retries=5):
    for i in range(retries):
        try:
            r = requests.get(url, timeout=10)
        except requests.RequestException:
            time.sleep(2 ** i)
            continue
        if r.status_code == 200:
            return r.json()
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(2 ** i)
            continue
        return None  # 400, 401, 403, 404 : pas la peine de réessayer
    return None


def resume(data):
    p = {s["stat"]["name"]: s["base_stat"] for s in data["stats"]}
    p["name"] = data["name"]
    p["id"] = data["id"]
    p["types"] = [t["type"]["name"] for t in data["types"]]
    p["taille"] = data["height"] / 10
    p["poids"] = data["weight"] / 10
    p["image"] = data["sprites"]["front_default"]
    p["total"] = sum(p[s] for s in STATS)
    return p


# cache d'1h (Q3)
@st.cache_data(ttl=3600)
def get_pokemon(nom):
    if not nom.strip():
        return None
    data = safe_get(f"{URL}/pokemon/{nom.lower().strip()}")
    return resume(data) if data else None


@st.cache_data(ttl=3600)
def get_type(t):
    return safe_get(f"{URL}/type/{t}")


@st.cache_data(ttl=3600)
def pokedex(debut, fin):
    # Q2 : on charge tout en parallèle
    async def fetch(session, sem, i):
        async with sem:
            for essai in range(5):
                try:
                    async with session.get(f"{URL}/pokemon/{i}") as r:
                        if r.status == 200:
                            return resume(await r.json())
                        if r.status not in (429, 500, 502, 503, 504):
                            return None
                except aiohttp.ClientError:
                    pass
                await asyncio.sleep(2 ** essai)

    async def main():
        sem = asyncio.Semaphore(50)
        async with aiohttp.ClientSession() as session:
            return await asyncio.gather(*[fetch(session, sem, i) for i in range(debut, fin + 1)])

    return [p for p in asyncio.run(main()) if p]


def tableau(lignes, colonnes):
    # tableau en markdown (st.dataframe ne marche pas chez moi à cause de pyarrow)
    txt = "| " + " | ".join(colonnes) + " |\n|" + "---|" * len(colonnes) + "\n"
    for l in lignes:
        txt += "| " + " | ".join(str(x) for x in l) + " |\n"
    st.markdown(txt, unsafe_allow_html=True)


def combat(p1, p2):
    degats = {p1["name"]: 0, p2["name"]: 0}
    log = []
    for tour in range(1, 6):
        for att, dfs in [(p1, p2), (p2, p1)]:
            d = round(att["attack"] / dfs["defense"] * 20 * random.uniform(0.85, 1))
            degats[att["name"]] += d
            log.append([tour, att["name"], dfs["name"], d])
    return degats, log


st.title("PokéAPI - Mini projet 2")
onglets = st.tabs(["Pokédex", "Fiche", "Comparaison", "Type", "Combat"])

with onglets[0]:
    debut, fin = st.slider("Numéros", 1, 1025, (1, 151))
    types = st.multiselect("Type", ["normal", "fire", "water", "grass", "electric", "ice", "fighting", "poison",
                                    "ground", "flying", "psychic", "bug", "rock", "ghost", "dragon", "dark",
                                    "steel", "fairy"])
    tri = st.selectbox("Trier par", ["total"] + STATS + ["taille", "poids", "id"])
    croissant = st.checkbox("Ordre croissant")

    liste = pokedex(debut, fin)
    if types:
        liste = [p for p in liste if any(t in p["types"] for t in types)]
    liste = sorted(liste, key=lambda p: p[tri], reverse=not croissant)

    st.write(f"{len(liste)} pokémons")
    tableau([[f"<img src='{p['image']}' width='40'>", p["id"], p["name"], ", ".join(p["types"]), p["taille"],
              p["poids"]] + [p[s] for s in STATS] + [p["total"]] for p in liste],
            ["", "id", "nom", "types", "taille", "poids", "pv", "att", "def", "att spé", "def spé", "vitesse", "total"])

with onglets[1]:
    nom = st.text_input("Nom ou id", "pikachu")
    p = get_pokemon(nom)
    if p is None:
        st.error("Pokémon introuvable")
    else:
        st.image(p["image"])
        st.write(f"**{p['name']}** - {', '.join(p['types'])} - {p['taille']} m - {p['poids']} kg")
        cols = st.columns(6)
        for c, s in zip(cols, STATS):
            c.metric(s, p[s])

with onglets[2]:
    c1, c2 = st.columns(2)
    a = get_pokemon(c1.text_input("Pokémon 1", "charizard"))
    b = get_pokemon(c2.text_input("Pokémon 2", "blastoise"))
    if a and b:
        for s in STATS + ["total"]:
            c1.metric(f"{a['name']} - {s}", a[s], a[s] - b[s])
            c2.metric(f"{b['name']} - {s}", b[s], b[s] - a[s])
        st.write("Plus de PV :", a["name"] if a["hp"] > b["hp"] else b["name"])
        st.write("Plus d'attaque :", a["name"] if a["attack"] > b["attack"] else b["name"])

with onglets[3]:
    t = st.text_input("Type", "fire")
    data = get_type(t)
    if data is None:
        st.error("Type introuvable")
    else:
        noms = [x["pokemon"]["name"] for x in data["pokemon"]]
        pvs = [get_pokemon(n)["hp"] for n in noms]
        st.metric("Nombre de pokémons", len(noms))
        st.metric("PV moyen", round(sum(pvs) / len(pvs), 1))

with onglets[4]:
    c1, c2 = st.columns(2)
    n1 = c1.text_input("Combattant 1", "pikachu")
    n2 = c2.text_input("Combattant 2", "gyarados")
    if st.button("Combat !"):
        p1, p2 = get_pokemon(n1), get_pokemon(n2)
        if p1 and p2:
            degats, log = combat(p1, p2)
            tableau(log, ["tour", "attaquant", "défenseur", "dégâts"])
            st.write("Dégâts totaux :", degats)
            st.success("Gagnant : " + max(degats, key=degats.get))


Overwriting app_pokemon.py


In [20]:
# arrêter le serveur local
await runner.cleanup()